In [1]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
import numpy as np
import os

D:\Users\anast\PycharmProjects\ProphetForecast\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri(
    "sqlite:///D:/Users/anast/PycharmProjects/ProphetForecast/mlflow.db")

mlflow.set_experiment("Timeseries Forecasting Project")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("Timeseries Forecasting Project"))

Tracking URI: sqlite:///D:/Users/anast/PycharmProjects/ProphetForecast/mlflow.db
Experiment: <Experiment: artifact_location='file:D:/Users/anast/PycharmProjects/ProphetForecast/mlruns/1', creation_time=1782894283941, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782894283941, lifecycle_stage='active', name='Timeseries Forecasting Project', tags={}, trace_location=None, workspace='default'>


In [3]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "Tuned XGBoost",
        "ARIMA",
        "Exponential Smoothing",
        "SARIMA",
        "Prophet",
        "Tuned LSTM"
    ],
    "MAE": [
        90.85,
        88.88,
        91.40,
        89.77,
        144.33,
        98.43,
        98.89,
        98.36,
        141.30
    ],
    "RMSE": [
        134.22,
        141.29,
        143.47,
        136.31,
        186.45,
        150.41,
        151.02,
        150.47,
        188.45
    ],
    "R2": [
        0.5028,
        0.4490,
        0.4319,
        0.4872,
        0.0405,
        0.3756,
        0.3705,
        0.3751,
        0.0198
    ]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,90.85,134.22,0.5028
1,Random Forest,88.88,141.29,0.4490
2,XGBoost,91.40,143.47,0.4319
3,Tuned XGBoost,89.77,136.31,0.4872
4,ARIMA,144.33,186.45,0.0405
5,Exponential Smoothing,98.43,150.41,0.3756
6,SARIMA,98.89,151.02,0.3705
7,Prophet,98.36,150.47,0.3751
8,Tuned LSTM,141.30,188.45,0.0198


In [4]:
for _, row in results.iterrows():

    with mlflow.start_run(run_name=row["Model"]):

        mlflow.log_param("model_name", row["Model"])

        mlflow.log_metric("MAE", row["MAE"])
        mlflow.log_metric("RMSE", row["RMSE"])
        mlflow.log_metric("R2", row["R2"])

        if row["Model"] == "Tuned XGBoost":
            mlflow.log_params({
                "n_estimators": 200,
                "max_depth": 2,
                "learning_rate": 0.08781417857605588,
                "subsample": 0.6138813743427279,
                "colsample_bytree": 0.9819211786756363
            })

        if row["Model"] == "Tuned LSTM":
            mlflow.log_param("tuning_method", "Keras Tuner")
            mlflow.log_param("max_trials", 10)
            mlflow.log_param("epochs", 30)
            mlflow.log_param("batch_size", 32)

print("All model results logged successfully!")
print("Logged runs:", len(results))

2026/07/01 11:36:59 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/07/01 11:36:59 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

All model results logged successfully!
Logged runs: 9


In [5]:
runs = mlflow.search_runs()
runs[["tags.mlflow.runName", "metrics.MAE", "metrics.RMSE", "metrics.R2"]]

,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2
0,Tuned LSTM,141.30,188.45,0.0198
1,Prophet,98.36,150.47,0.3751
2,SARIMA,98.89,151.02,0.3705
3,Exponential Smoothing,98.43,150.41,0.3756
4,ARIMA,144.33,186.45,0.0405
...,...,...,...,...
101,Tuned XGBoost,89.77,136.31,0.4872
102,Tuned XGBoost,NaN,NaN,NaN
103,Tuned XGBoost,NaN,NaN,NaN
104,Tuned XGBoost,NaN,NaN,NaN


In [6]:
results

,Model,MAE,RMSE,R2
0,Linear Regression,90.85,134.22,0.5028
1,Random Forest,88.88,141.29,0.4490
2,XGBoost,91.40,143.47,0.4319
3,Tuned XGBoost,89.77,136.31,0.4872
4,ARIMA,144.33,186.45,0.0405
5,Exponential Smoothing,98.43,150.41,0.3756
6,SARIMA,98.89,151.02,0.3705
7,Prophet,98.36,150.47,0.3751
8,Tuned LSTM,141.30,188.45,0.0198


In [7]:
import mlflow

runs = mlflow.search_runs()
print(runs.shape)
runs

(106, 23)


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.RMSE,metrics.MAE,metrics.R2,params.epochs,...,params.max_trials,params.subsample,params.max_depth,params.n_estimators,params.colsample_bytree,params.learning_rate,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.name,tags.mlflow.source.type
0,ae19eb206e8c4a56b13c5280c4e55f2c,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 09:37:07.098000+00:00,2026-07-01 09:37:08.408000+00:00,188.45,141.30,0.0198,30,...,10,None,None,None,None,None,anast,Tuned LSTM,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
1,1770880d09444f45b422760029da2b7f,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 09:37:05.862000+00:00,2026-07-01 09:37:06.922000+00:00,150.47,98.36,0.3751,None,...,None,None,None,None,None,None,anast,Prophet,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
2,94c2b89570dc4cb0ba2c26488c11d0c1,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 09:37:04.844000+00:00,2026-07-01 09:37:05.687000+00:00,151.02,98.89,0.3705,None,...,None,None,None,None,None,None,anast,SARIMA,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
3,d7c3b4c5021d4640a3c693ba9fca4760,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 09:37:04.074000+00:00,2026-07-01 09:37:04.716000+00:00,150.41,98.43,0.3756,None,...,None,None,None,None,None,None,anast,Exponential Smoothing,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
4,d244183b20d24677bc8cad286b896e7a,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 09:37:03.205000+00:00,2026-07-01 09:37:03.924000+00:00,186.45,144.33,0.0405,None,...,None,None,None,None,None,None,anast,ARIMA,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,678bc5349e094256be8bb78719588611,1,FINISHED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 08:40:50.473000+00:00,2026-07-01 08:40:51.245000+00:00,136.31,89.77,0.4872,None,...,None,0.6138813743427279,2,200,0.9819211786756363,0.08781417857605588,anast,Tuned XGBoost,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
102,7f20d7f52d2a42fba513f6906598cb6a,1,FAILED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 08:32:08.050000+00:00,2026-07-01 08:32:08.216000+00:00,NaN,NaN,NaN,None,...,None,None,None,None,None,None,anast,Tuned XGBoost,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
103,f4b489b253744ca5a423daed01468bf4,1,FAILED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 08:28:41.555000+00:00,2026-07-01 08:28:41.743000+00:00,NaN,NaN,NaN,None,...,None,None,None,None,None,None,anast,Tuned XGBoost,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK
104,c91853157dce4af0a731b6ccd3449890,1,FAILED,file:D:/Users/anast/PycharmProjects/ProphetFor...,2026-07-01 08:28:36.928000+00:00,2026-07-01 08:28:37.060000+00:00,NaN,NaN,NaN,None,...,None,None,None,None,None,None,anast,Tuned XGBoost,D:\Users\anast\PycharmProjects\ProphetForecast...,NOTEBOOK


In [9]:
mlflow.set_tracking_uri(
    "sqlite:///D:/Users/anast/PycharmProjects/ProphetForecast/mlflow.db"
)

mlflow.set_experiment("Timeseries Forecasting Project")

for _, row in results.iterrows():
    with mlflow.start_run(run_name=row["Model"]):
        mlflow.log_param("model_name", row["Model"])
        mlflow.log_metric("MAE", float(row["MAE"]))
        mlflow.log_metric("RMSE", float(row["RMSE"]))
        mlflow.log_metric("R2", float(row["R2"]))

print("Runs logged again.")

Runs logged again.


In [10]:
results.to_csv("model_results.csv", index=False)

Although Linear Regression achieved the lowest RMSE and the highest R², visual inspection of the forecast revealed unrealistic long-term extrapolation. Since forecasting models should not only minimize prediction error but also produce plausible future trends, the tuned XGBoost model was selected as the champion model. It achieved competitive error metrics while generating substantially more realistic forecasts.